# Small Multiples

> *Compared to what?* — Edward Tufte

In 1878, Eadweard Muybridge settled a bet with a series of photographs: **Horse in Motion**. Each frame shows the same subject at a different instant, on the same scale — an early [small multiple](https://en.wikipedia.org/wiki/Small_multiple) (also called a trellis, lattice, or panel chart).


![Horse In Motion, Muybridge (1886)](../assets/seaborn/horse_in_motion.png)


In this lesson you will learn how seaborn's **figure-level** functions build that idea into your plots:

- how `displot`, `relplot`, and `catplot` wrap the axes-level functions
- how the `kind=` parameter selects the underlying plot (histogram, KDE, scatter, …),
- how `col`, `row`, and `hue` split the data into **facets** — small multiples that answer Tufte's question side by side.

**Learning objectives**

- Distinguish axes-level functions (`histplot`, `kdeplot`, `scatterplot`) from figure-level ones (`displot`, `relplot`, `catplot`).
- Produce the same distribution and relationship plots using `kind=` on a figure-level function.
- Build faceted grids with `col`, `row`, and `hue`.
- Control subplot size with `height` and `aspect`, and tweak the returned `FacetGrid`.


## Setup


In [ ]:
# pandas for tabular data: https://pandas.pydata.org/docs/
import pandas as pd

# matplotlib is the underlying plotting engine: https://matplotlib.org/stable/
import matplotlib.pyplot as plt

# seaborn is a high-level statistical plotting library: https://seaborn.pydata.org/
import seaborn as sns

# Apply seaborn's default theme (colors, fonts, grid) to every matplotlib figure.
sns.set_theme()

# load_dataset downloads (and caches) a tidy example DataFrame.
penguins = sns.load_dataset("penguins")

penguins.head()


# Figure-level functions

In [Overview of seaborn plotting functions](https://seaborn.pydata.org/tutorial/function_overview.html), seaborn groups similar plots into **modules**. Each module has several **axes-level** functions (they draw on one `Axes`) and one **figure-level** function that wraps them:

| Module | Figure-level | Axes-level examples |
|--------|--------------|---------------------|
| Relational | `relplot` | `scatterplot`, `lineplot` |
| Distributional | `displot` | `histplot`, `kdeplot`, `ecdfplot` |
| Categorical | `catplot` | `countplot`, `boxplot`, `barplot`, … |

![](../assets/seaborn/seaborn_figure-level_functions.png)

The figure-level function is the single entry point for that module. It forwards your arguments to the right axes-level function behind the scenes.


## Axes-level vs figure-level

The **object-oriented** matplotlib pattern:

- create a `Figure` and `Axes` with `plt.subplots()`,
- pass `ax=ax` into `sns.histplot`, `sns.kdeplot`, or `sns.scatterplot`.

Those are **axes-level** functions: they only change the axes you give them, so you can place several different plot types on one figure.

**Figure-level** functions work differently:

- they create and own an entire figure (usually a `FacetGrid`),
- you do **not** pass `ax=` — there is no "current" axes to share with another plot,
- the legend is often placed **outside** the plot area,
- the function returns a grid object (e.g. `FacetGrid`) for further tweaking.

The main payoff is **faceting**: repeating the same plot for each level of `col`, `row`, or `hue` — our small multiples.


## The `kind=` parameter

A figure-level function does not replace the axes-level API; it **dispatches** to it via `kind=`.

- `sns.displot(..., kind="hist")` → same drawing code as `sns.histplot`
- `sns.displot(..., kind="kde")` → same as `sns.kdeplot`
- `sns.relplot(..., kind="scatter")` → same as `sns.scatterplot`

Kind-specific options (e.g. `binwidth` for histograms) are passed through but may not appear in the figure-level function's signature — check the axes-level docs when you need them.

### Distribution: histogram via `displot`

In `01_visualization.ipynb` we used `histplot` on a single axes. The figure-level twin is `displot` with `kind="hist"`:


In [ ]:
grid = sns.displot(
    data=penguins,
    kind="hist",       # use histplot under the hood
    x="body_mass_g",
    binwidth=200,      # same knob as in 01_visualization.ipynb
    height=3,
    aspect=1.5,
)

grid.figure.suptitle("Body mass — histogram via displot", y=1.02)


### Distribution: KDE via `displot`

Switch the representation with one argument — no need to import a different function:


In [ ]:
grid = sns.displot(
    data=penguins,
    kind="kde",        # use kdeplot under the hood
    x="body_mass_g",
    fill=True,
    height=3,
    aspect=1.5,
)

grid.figure.suptitle("Body mass — KDE via displot", y=1.02)


### Relationship: scatter via `relplot`

`01_visualization.ipynb` used `scatterplot` with `ax=`. The figure-level version is `relplot` with `kind="scatter"`:


In [ ]:
grid = sns.relplot(
    data=penguins,
    kind="scatter",    # use scatterplot under the hood
    x="flipper_length_mm",
    y="body_mass_g",
    hue="species",
    height=3.5,
    aspect=1.2,
)

grid.figure.suptitle("Body mass vs. flipper length — scatter via relplot", y=1.02)


# Small multiples (faceting)

A [small multiple](https://en.wikipedia.org/wiki/Small_multiple) is a series of similar charts on the **same scale and axes**, so you can compare partitions of the dataset at a glance. seaborn calls this **faceting**; the tutorial section [Conditional small multiples](https://seaborn.pydata.org/tutorial/axis_grids.html#conditional-small-multiples) explains the underlying `FacetGrid`.

Three faceting variables:

- `col` — one subplot column per category level
- `row` — one subplot row per category level
- `hue` — color encodes a third variable (like a depth axis through the same panels)

You met a preview in `01_visualization.ipynb`; here we build the idea systematically.


## One column of facets: `col=`

Instead of stacking three species' KDE curves in one axes (`01_visualization.ipynb`'s `hue=` on `kdeplot`), put each species in its own panel:


In [ ]:
grid = sns.displot(
    data=penguins,
    kind="kde",
    x="body_mass_g",
    hue="sex",         # color curves by sex within each panel
    fill=True,
    col="species",     # one small multiple per species
    height=3,
    aspect=0.9,
)

grid.figure.suptitle("Body mass KDE by species (columns)", y=1.02)


## Relational facets: `relplot` with `col=`

The same flipper-length vs body-mass scatter from `01_visualization.ipynb`, but one panel per island:


In [ ]:
grid = sns.relplot(
    data=penguins,
    kind="scatter",
    x="flipper_length_mm",
    y="body_mass_g",
    hue="species",
    col="island",      # facet by island — small multiples
    height=3,
    aspect=0.9,
)

grid.figure.suptitle("Body mass vs. flipper length, faceted by island", y=1.02)


## A row-and-column grid: `row=` and `col=`

You can facet on two categorical variables at once. Each cell shows the same plot for one `(row, col)` combination:


In [ ]:
grid = sns.displot(
    data=penguins,
    kind="hist",
    x="body_mass_g",
    binwidth=250,
    hue="sex",
    multiple="stack",
    row="species",     # one row per species
    col="island",      # one column per island
    height=2.5,
    aspect=1.1,
)

grid.figure.suptitle("Body mass histograms in a species × island grid", y=1.02)


## Many levels: `col_wrap`

When one variable has many categories, `col_wrap` flows panels onto multiple rows instead of squeezing them into one long row (you cannot combine `col_wrap` with `row`):


In [ ]:
grid = sns.displot(
    data=penguins,
    kind="kde",
    x="flipper_length_mm",
    hue="species",
    col="island",
    col_wrap=2,        # at most 2 columns, then wrap to the next row
    height=2.5,
    aspect=1.0,
)

grid.figure.suptitle("Flipper length KDE — islands wrapped in a 2-column grid", y=1.02)


# Controlling figure size

With axes-level plots, you set the overall figure size through matplotlib (`figsize` in `plt.subplots()` or `Figure.set_size_inches()`).

Figure-level functions use **`height`** and **`aspect`** instead:

- they refer to the size of **each subplot**, not the whole figure,
- width of one facet ≈ `height * aspect`,
- adding more `col` or `row` levels **grows** the figure so each facet keeps the same shape.

Compare two aspect ratios for the same faceted `relplot`:


In [ ]:
# Wide facets: aspect > 1 stretches each panel horizontally.
grid_wide = sns.relplot(
    data=penguins,
    kind="scatter",
    x="flipper_length_mm",
    y="body_mass_g",
    col="island",
    height=3,
    aspect=1.4,
)
grid_wide.figure.suptitle("Taller, wider facets (height=3, aspect=1.4)", y=1.02)


In [ ]:
# Tall, narrow facets: aspect < 1.
grid_narrow = sns.relplot(
    data=penguins,
    kind="scatter",
    x="flipper_length_mm",
    y="body_mass_g",
    col="island",
    height=3,
    aspect=0.7,
)
grid_narrow.figure.suptitle("Same height, narrower facets (aspect=0.7)", y=1.02)


# Customizing the grid

Figure-level functions return a **`FacetGrid`** (stored in `grid`). Useful methods:

- `grid.figure.suptitle(...)` — title above the whole figure
- `grid.set_axis_labels(x_label, y_label)` — label exterior axes once
- `grid.set(xlim=..., yticks=...)` — broadcast matplotlib settings to every facet
- `grid.add_legend()` — if you used `hue` and the legend is missing

Example after a faceted scatter:


In [ ]:
grid = sns.relplot(
    data=penguins,
    kind="scatter",
    x="flipper_length_mm",
    y="body_mass_g",
    hue="species",
    col="island",
    height=3,
    aspect=0.9,
)

# FacetGrid helpers (not part of the vanilla matplotlib Axes API).
grid.set_axis_labels("Flipper length (mm)", "Body mass (g)")
grid.set(xlim=(170, 240), ylim=(2500, 6500))

grid.figure.suptitle("Faceted scatter with shared axis labels", y=1.05)


# Figure-level vs axes-level: which to use?

From the [seaborn function overview](https://seaborn.pydata.org/tutorial/function_overview.html):

**Figure-level advantages**

- Easy faceting with `col`, `row`, and `hue`
- Legend outside the plot by default
- `height` / `aspect` keep facet size consistent as you add columns or rows
- One-liner customization via `FacetGrid` methods

**Figure-level drawbacks**

- Kind-specific parameters are easy to miss (they live on `histplot`, `kdeplot`, etc.)
- Cannot drop a figure-level plot onto an existing `ax` in a larger composite figure
- Slightly different sizing rules than matplotlib's `figsize`

**Practical rule**

- Use **figure-level** functions (`displot`, `relplot`, `catplot`) when you want faceted small multiples or a quick standalone chart.
- Use **axes-level** functions (`histplot`, `kdeplot`, `scatterplot`, …) when you are assembling a custom multi-panel figure with `plt.subplots()`.

## Summary

- seaborn modules each expose one **figure-level** function that wraps several **axes-level** functions; `kind=` picks which one runs.
- `displot` covers distributions (`hist`, `kde`, …); `relplot` covers relationships (`scatter`, `line`); `catplot` covers categorical plots.
- **`col`**, **`row`**, and **`hue`** build [small multiples](https://en.wikipedia.org/wiki/Small_multiple) — the same plot repeated for subsets of the data.
- Control per-facet size with **`height`** and **`aspect`**; tune the result through the returned **`FacetGrid`**.

You now have the full arc of L3 visualization: distributions and relationships (`01_visualization.ipynb`), then faceted comparison (`02_small_multiples.ipynb`).
